In [ ]:
!pip install kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle competitions download -c optiver-realized-volatility-prediction
!unzip optiver-realized-volatility-prediction.zip


cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 4, in <module>
    from kaggle.cli import main
  File "/usr/local/lib/python3.12/dist-packages/kaggle/__init__.py", line 6, in <module>
    api.authenticate()
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 434, in authenticate
    raise IOError('Could not find {}. Make sure it\'s located in'
OSError: Could not find kaggle.json. Make sure it's located in /root/.kaggle. Or use the environment method. See setup instructions at https://github.com/Kaggle/kaggle-api/
unzip:  cannot find or open optiver-realized-volatility-prediction.zip, optiver-realized-volatility-prediction.zip.zip or optiver-realized-volatility-prediction.zip.ZIP.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Multi-stock memory-optimized streaming training for Optiver + STFPool CNN
# Requires: pyarrow, pandas, torch
# Mount Drive first in Colab:
# from google.colab import drive; drive.mount('/content/drive')

import os
from pathlib import Path
import random
import time
import collections

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

# -------------------------
# Config (change these)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/Fuzzy_pool/optiver-realized-volatility-prediction"
BOOK_PATH = Path(DATA_DIR) / "book_train.parquet"
TRAIN_CSV = Path(DATA_DIR) / "train.csv"   # used to get stock_id list / real targets
LOOKBACK = 60
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
MAX_TIME_IDS_PER_STOCK = None   # keep None for full per-stock; set small for quick debug (e.g., 2000)
NUM_STOCKS = None               # None => use all stocks in train.csv; or set e.g. 20 for subset
USE_REAL_TARGETS = True         # if True, will join train.csv to use realized_volatility; else uses simple next-mid label
EDA_NOTEBOOK_PATH = "/mnt/data/optiver-realized-volatility-prediction-eda.ipynb"  # your uploaded EDA

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)
print("BOOK_PATH:", BOOK_PATH)
print("EDA notebook (for reference):", EDA_NOTEBOOK_PATH)

# -------------------------
# Helpers: load stock_id list and optional targets
# -------------------------
def get_stock_list(train_csv_path, num_stocks=None):
    df = pd.read_csv(train_csv_path)
    stock_ids = df['stock_id'].unique().tolist()
    stock_ids.sort()
    if num_stocks is not None:
        stock_ids = stock_ids[:num_stocks]
    return stock_ids

def load_targets_for_stock(train_csv_path, stock_id):
    # returns dict: time_id -> target
    df = pd.read_csv(train_csv_path)
    s = df[df.stock_id == stock_id][['time_id', 'target']].set_index('time_id')['target']
    return s.to_dict()

# -------------------------
# Read parquet for a single stock using pyarrow.dataset filter (memory efficient)
# -------------------------
def read_book_for_stock_pa(book_parquet_path: str, stock_id: int, columns=None, max_time_ids=None):
    ds_path = str(book_parquet_path)
    # Fix: Add partitioning='hive' to correctly interpret stock_id from directory names
    dataset = ds.dataset(ds_path, format="parquet", partitioning="hive")
    filt = (ds.field("stock_id") == stock_id)
    table = dataset.to_table(filter=filt, columns=columns)
    df = table.to_pandas()
    if df.shape[0] == 0:
        raise RuntimeError(f"No rows for stock_id={stock_id}")
    df = df.sort_values(["time_id", "seconds_in_bucket"]).reset_index(drop=True)
    if max_time_ids is not None:
        tids = df["time_id"].unique()[:max_time_ids]
        df = df[df["time_id"].isin(tids)].reset_index(drop=True)
    return df

# -------------------------
# Aggregate per time_id -> features (same small feature set)
# -------------------------
def aggregate_book_to_time_features(df):
    df = df.copy()
    # compute level-1 derived features
    # avoid division by zero
    denom = (df["bid_size1"] + df["ask_size1"] + 1e-9)
    df["wap1"] = (df["bid_price1"] * df["ask_size1"] + df["ask_price1"] * df["bid_size1"]) / denom
    df["spread1"] = df["ask_price1"] - df["bid_price1"]
    df["mid1"] = (df["ask_price1"] + df["bid_price1"]) / 2.0
    df["liq_imb1"] = (df["bid_size1"] - df["ask_size1"]) / denom

    agg = df.groupby("time_id").agg({
        "wap1": ["mean", "std"],
        "spread1": ["mean", "std"],
        "mid1": ["mean"],
        "liq_imb1": ["mean"]
    })
    agg.columns = ["_".join(col).strip() for col in agg.columns.values]
    agg = agg.reset_index().sort_values("time_id").reset_index(drop=True)
    return agg

# -------------------------
# IterableDataset streaming windows for a single stock, yields (window_tensor, label)
# If USE_REAL_TARGETS True, joins train.csv target; else uses thresholded next-mid
# -------------------------
class StockWindowStream(IterableDataset):
    def __init__(self, book_parquet_path, stock_id, lookback=LOOKBACK, max_time_ids=None, normalize=True, train_csv_path=None):
        self.book_parquet_path = book_parquet_path
        self.stock_id = stock_id
        self.lookback = lookback
        self.max_time_ids = max_time_ids
        self.normalize = normalize
        self.train_csv_path = train_csv_path

    def __iter__(self):
        # 1) read stock chunk
        df = read_book_for_stock_pa(self.book_parquet_path, self.stock_id, max_time_ids=self.max_time_ids)
        feats = aggregate_book_to_time_features(df)   # time_id indexed features
        # 2) compute labels
        if self.train_csv_path is not None and USE_REAL_TARGETS:
            # join real realized volatility targets from train.csv
            train_df = pd.read_csv(self.train_csv_path)
            s = train_df[train_df.stock_id == self.stock_id].set_index('time_id')['target']
            # create labels: bucketize target into binary (above median -> 1 else 0) as example
            tvals = feats['time_id'].map(s).fillna(0.0).values.astype(np.float32)
            # threshold at median (you can choose regression instead)
            med = np.median(tvals)
            labels = (tvals > med).astype(np.int64)
        else:
            # simple next-mid threshold approach
            mid = feats["mid1_mean"].values.astype(np.float32)
            next_ret = np.empty_like(mid)
            next_ret[:-1] = mid[1:] - mid[:-1]
            next_ret[-1] = 0.0
            thr = np.std(next_ret) * 0.01
            labels = (next_ret > thr).astype(np.int64)

        cols = [c for c in feats.columns if c != "time_id"]
        arr = feats[cols].astype(np.float32).values  # small array (n_time x n_features)
        dq = collections.deque(maxlen=self.lookback)

        # stream rows
        for i, row in enumerate(arr):
            dq.append(row)
            if len(dq) == self.lookback:
                win = np.stack(dq, axis=1)  # shape (n_features, lookback)
                if self.normalize:
                    mu = win.mean(axis=1, keepdims=True)
                    sd = win.std(axis=1, keepdims=True) + 1e-9
                    win = (win - mu) / sd
                label_index = i - (self.lookback - 1)
                lbl = int(labels[label_index])
                yield torch.tensor(win, dtype=torch.float32), lbl

# -------------------------
# Minimal STFPool + small model (same as before)
# -------------------------
class STFPool2d(nn.Module):
    def __init__(self, kernel_size=3, stride=2, num_terms=3):
        super().__init__()
        self.k = kernel_size
        self.stride = stride
        self.num_terms = num_terms
        self.pos_weights = nn.Parameter(torch.ones(1, 1, self.k, self.k))
        self.tfn_logits = nn.Parameter(torch.randn(num_terms, 3) * 0.05)

    def forward(self, x):
        B, C, H, W = x.shape
        k = self.k
        x_p = x.unfold(2, k, self.stride).unfold(3, k, self.stride)
        mask = self._geom_mask(k, x.device) * self.pos_weights
        x_masked = x_p * mask
        n = k*k
        x_flat = x_masked.contiguous().view(B, C, x_p.shape[2], x_p.shape[3], n)

        raw = F.softplus(self.tfn_logits)
        a = raw[:,0]; b = a + raw[:,1]; c = b + raw[:,2]

        x_ = x_flat.unsqueeze(-2)
        a_ = a.view(1,1,1,1,-1,1); b_ = b.view(1,1,1,1,-1,1); c_ = c.view(1,1,1,1,-1,1)
        left = torch.where((x_ > a_) & (x_ <= b_), (x_ - a_) / (b_ - a_ + 1e-9), torch.zeros_like(x_))
        right = torch.where((x_ > b_) & (x_ < c_), (c_ - x_) / (c_ - b_ + 1e-9), torch.zeros_like(x_))
        mus = left + right

        w = mask.view(1,1,1,1,n)
        agg = (mus * w).sum(-1) / (w.sum(-1) + 1e-9)
        centroids = b.view(1,1,1,1,-1)
        out = (agg * centroids).sum(-1) / (agg.sum(-1) + 1e-9)
        return out

    def _geom_mask(self, k, device):
        mask = torch.zeros(k, k, device=device)
        for i in range(k):
            mask[i, :k - i] = 1.0
        return mask.view(1, 1, k, k)

class STFNetSmall(nn.Module):
    def __init__(self, in_channels=1, base_filters=32, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, base_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(base_filters)
        self.stf1 = STFPool2d(kernel_size=3, stride=2)
        self.conv2 = nn.Conv2d(base_filters, base_filters*2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(base_filters*2)
        # Fix: Change kernel_size for stf2 to 2 as the input height dimension will be 2 after previous layers
        self.stf2 = STFPool2d(kernel_size=2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(base_filters*2, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.stf1(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.stf2(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

# -------------------------
# Collate: create batch tensor
# -------------------------
def collate_fn(batch):
    windows, labels = zip(*batch)
    xs = torch.stack([w for w in windows], dim=0)  # B, H, W
    xs = xs.unsqueeze(1)  # B,1,H,W
    ys = torch.tensor(labels, dtype=torch.long)
    return xs, ys

# -------------------------
# Multi-stock training loop (streams stock by stock)
# -------------------------
def train_multi_stock(book_path, train_csv_path, epochs=EPOCHS, batch_size=BATCH_SIZE,
                      lookback=LOOKBACK, max_time_ids=MAX_TIME_IDS_PER_STOCK, num_stocks=NUM_STOCKS):
    stock_list = get_stock_list(train_csv_path, num_stocks)
    print("Training on stock_count =", len(stock_list))

    model = STFNetSmall(in_channels=1).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()

    for ep in range(1, epochs+1):
        random.shuffle(stock_list)
        ep_loss = 0.0
        ep_correct = 0
        ep_total = 0
        t0 = time.time()

        for sid in stock_list:
            # create streaming dataset for this stock
            ds = StockWindowStream(str(book_path), sid, lookback=lookback, max_time_ids=max_time_ids,
                                   normalize=True, train_csv_path=train_csv_path if USE_REAL_TARGETS else None)
            loader = DataLoader(ds, batch_size=batch_size, collate_fn=collate_fn, num_workers=NUM_WORKERS)

            # iterate batches for this stock
            for xb, yb in loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                preds = model(xb)
                loss = crit(preds, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()

                ep_loss += loss.item() * xb.size(0)
                ep_correct += (preds.argmax(1) == yb).sum().item()
                ep_total += xb.size(0)

        if ep_total == 0:
            print("No training samples found. Check lookback/max_time_ids.")
            break

        print(f"Epoch {ep}/{epochs}  loss={ep_loss/ep_total:.4f} acc={ep_correct/ep_total:.4f} time={time.time()-t0:.1f}s")

    return model

# -------------------------
# Run training (example)
# -------------------------
if __name__ == "__main__":
    # Mount drive in Colab before running
    # from google.colab import drive; drive.mount('/content/drive')

    model = train_multi_stock(BOOK_PATH, TRAIN_CSV, epochs=EPOCHS, batch_size=BATCH_SIZE,
                              lookback=LOOKBACK, max_time_ids=MAX_TIME_IDS_PER_STOCK, num_stocks=NUM_STOCKS)
    # Save model
    torch.save(model.state_dict(), "stf_multi_stock.pth")
    print("Saved model: stf_multi_stock.pth")


Device: cuda
BOOK_PATH: /content/drive/MyDrive/Fuzzy_pool/optiver-realized-volatility-prediction/book_train.parquet
EDA notebook (for reference): /mnt/data/optiver-realized-volatility-prediction-eda.ipynb
Training on stock_count = 112
Epoch 1/10  loss=0.6067 acc=0.6604 time=597.1s
Epoch 2/10  loss=0.5678 acc=0.6929 time=349.6s
Epoch 3/10  loss=0.5695 acc=0.6885 time=349.9s
Epoch 4/10  loss=0.5866 acc=0.6693 time=346.6s
Epoch 5/10  loss=0.5938 acc=0.6603 time=348.8s
Epoch 6/10  loss=0.6039 acc=0.6465 time=345.3s
Epoch 7/10  loss=0.6289 acc=0.6232 time=347.7s
Epoch 8/10  loss=0.6383 acc=0.6068 time=342.8s
Epoch 9/10  loss=0.6311 acc=0.6078 time=343.1s
Epoch 10/10  loss=0.6480 acc=0.5849 time=347.3s
Saved model: stf_multi_stock.pth


In [ ]:
# Multi-stock memory-optimized streaming training for Optiver + TinyCNN
# Requires: pyarrow, pandas, torch
# Mount Drive first in Colab:
# from google.colab import drive; drive.mount('/content/drive')

import os
from pathlib import Path
import random
import time
import collections

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

# -------------------------
# Config (change these)
# -------------------------
DATA_DIR = "/content/drive/MyDrive/Fuzzy_pool/optiver-realized-volatility-prediction"
BOOK_PATH = Path(DATA_DIR) / "book_train.parquet"
TRAIN_CSV = Path(DATA_DIR) / "train.csv"   # used to get stock_id list / real targets
LOOKBACK = 60
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
MAX_TIME_IDS_PER_STOCK = None   # keep None for full per-stock; set small for quick debug (e.g., 2000)
NUM_STOCKS = None               # None => use all stocks in train.csv; or set e.g. 20 for subset
USE_REAL_TARGETS = True         # if True, will join train.csv to use realized_volatility; else uses simple next-mid label
EDA_NOTEBOOK_PATH = "/mnt/data/optiver-realized-volatility-prediction-eda.ipynb"  # your uploaded EDA (local path)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)
print("BOOK_PATH:", BOOK_PATH)
print("EDA notebook (for reference):", EDA_NOTEBOOK_PATH)

# -------------------------
# Helpers: load stock_id list and optional targets
# -------------------------
def get_stock_list(train_csv_path, num_stocks=None):
    df = pd.read_csv(train_csv_path)
    stock_ids = df['stock_id'].unique().tolist()
    stock_ids.sort()
    if num_stocks is not None:
        stock_ids = stock_ids[:num_stocks]
    return stock_ids

def load_targets_for_stock(train_csv_path, stock_id):
    # returns dict: time_id -> target
    df = pd.read_csv(train_csv_path)
    s = df[df.stock_id == stock_id][['time_id', 'target']].set_index('time_id')['target']
    return s.to_dict()

# -------------------------
# Read parquet for a single stock using pyarrow.dataset filter (memory efficient)
# -------------------------
def read_book_for_stock_pa(book_parquet_path: str, stock_id: int, columns=None, max_time_ids=None):
    ds_path = str(book_parquet_path)
    dataset = ds.dataset(ds_path, format="parquet", partitioning="hive")
    filt = (ds.field("stock_id") == stock_id)
    table = dataset.to_table(filter=filt, columns=columns)
    df = table.to_pandas()
    if df.shape[0] == 0:
        raise RuntimeError(f"No rows for stock_id={stock_id}")
    df = df.sort_values(["time_id", "seconds_in_bucket"]).reset_index(drop=True)
    if max_time_ids is not None:
        tids = df["time_id"].unique()[:max_time_ids]
        df = df[df["time_id"].isin(tids)].reset_index(drop=True)
    return df

# -------------------------
# Aggregate per time_id -> features (same small feature set)
# -------------------------
def aggregate_book_to_time_features(df):
    df = df.copy()
    # compute level-1 derived features
    denom = (df["bid_size1"] + df["ask_size1"] + 1e-9)
    df["wap1"] = (df["bid_price1"] * df["ask_size1"] + df["ask_price1"] * df["bid_size1"]) / denom
    df["spread1"] = df["ask_price1"] - df["bid_price1"]
    df["mid1"] = (df["ask_price1"] + df["bid_price1"]) / 2.0
    df["liq_imb1"] = (df["bid_size1"] - df["ask_size1"]) / denom

    agg = df.groupby("time_id").agg({
        "wap1": ["mean", "std"],
        "spread1": ["mean", "std"],
        "mid1": ["mean"],
        "liq_imb1": ["mean"]
    })
    agg.columns = ["_".join(col).strip() for col in agg.columns.values]
    agg = agg.reset_index().sort_values("time_id").reset_index(drop=True)
    return agg

# -------------------------
# IterableDataset streaming windows for a single stock, yields (window_tensor, label)
# -------------------------
class StockWindowStream(IterableDataset):
    def __init__(self, book_parquet_path, stock_id, lookback=LOOKBACK, max_time_ids=None, normalize=True, train_csv_path=None):
        self.book_parquet_path = book_parquet_path
        self.stock_id = stock_id
        self.lookback = lookback
        self.max_time_ids = max_time_ids
        self.normalize = normalize
        self.train_csv_path = train_csv_path

    def __iter__(self):
        # 1) read stock chunk
        df = read_book_for_stock_pa(self.book_parquet_path, self.stock_id, max_time_ids=self.max_time_ids)
        feats = aggregate_book_to_time_features(df)   # time_id indexed features
        # 2) compute labels
        if self.train_csv_path is not None and USE_REAL_TARGETS:
            train_df = pd.read_csv(self.train_csv_path)
            s = train_df[train_df.stock_id == self.stock_id].set_index('time_id')['target']
            tvals = feats['time_id'].map(s).fillna(0.0).values.astype(np.float32)
            med = np.median(tvals)
            labels = (tvals > med).astype(np.int64)
        else:
            mid = feats["mid1_mean"].values.astype(np.float32)
            next_ret = np.empty_like(mid)
            next_ret[:-1] = mid[1:] - mid[:-1]
            next_ret[-1] = 0.0
            thr = np.std(next_ret) * 0.01
            labels = (next_ret > thr).astype(np.int64)

        cols = [c for c in feats.columns if c != "time_id"]
        arr = feats[cols].astype(np.float32).values  # small array (n_time x n_features)
        dq = collections.deque(maxlen=self.lookback)

        for i, row in enumerate(arr):
            dq.append(row)
            if len(dq) == self.lookback:
                win = np.stack(dq, axis=1)  # shape (n_features, lookback)
                if self.normalize:
                    mu = win.mean(axis=1, keepdims=True)
                    sd = win.std(axis=1, keepdims=True) + 1e-9
                    win = (win - mu) / sd
                label_index = i - (self.lookback - 1)
                lbl = int(labels[label_index])
                yield torch.tensor(win, dtype=torch.float32), lbl

# -------------------------
# TinyCNN model (replaces STFNetSmall)
# -------------------------
class TinyCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        # Input: (B,1,H=features,W=lookback)
        self.conv1 = nn.Conv2d(in_channels, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.gpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        # x: (B,1,H,W)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)                 # downsample
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)                 # downsample
        x = self.gpool(x).view(x.size(0), -1)
        return self.fc(x)

# -------------------------
# Collate: create batch tensor
# -------------------------
def collate_fn(batch):
    windows, labels = zip(*batch)
    xs = torch.stack([w for w in windows], dim=0)  # B, H, W
    xs = xs.unsqueeze(1)  # B,1,H,W
    ys = torch.tensor(labels, dtype=torch.long)
    return xs, ys

# -------------------------
# Multi-stock training loop (streams stock by stock) using TinyCNN
# -------------------------
def train_multi_stock(book_path, train_csv_path, epochs=EPOCHS, batch_size=BATCH_SIZE,
                      lookback=LOOKBACK, max_time_ids=MAX_TIME_IDS_PER_STOCK, num_stocks=NUM_STOCKS):
    stock_list = get_stock_list(train_csv_path, num_stocks)
    print("Training on stock_count =", len(stock_list))

    model = TinyCNN(in_channels=1).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()

    for ep in range(1, epochs+1):
        random.shuffle(stock_list)
        ep_loss = 0.0
        ep_correct = 0
        ep_total = 0
        t0 = time.time()

        for sid in stock_list:
            ds = StockWindowStream(str(book_path), sid, lookback=lookback, max_time_ids=max_time_ids,
                                   normalize=True, train_csv_path=train_csv_path if USE_REAL_TARGETS else None)
            loader = DataLoader(ds, batch_size=batch_size, collate_fn=collate_fn, num_workers=NUM_WORKERS)

            for xb, yb in loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                preds = model(xb)
                loss = crit(preds, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()

                ep_loss += loss.item() * xb.size(0)
                ep_correct += (preds.argmax(1) == yb).sum().item()
                ep_total += xb.size(0)

        if ep_total == 0:
            print("No training samples found. Check lookback/max_time_ids.")
            break

        print(f"Epoch {ep}/{epochs}  loss={ep_loss/ep_total:.4f} acc={ep_correct/ep_total:.4f} time={time.time()-t0:.1f}s")

    return model

# -------------------------
# Run training (example)
# -------------------------
if __name__ == "__main__":
    # Mount drive in Colab before running:
    # from google.colab import drive; drive.mount('/content/drive')

    model = train_multi_stock(BOOK_PATH, TRAIN_CSV, epochs=EPOCHS, batch_size=BATCH_SIZE,
                              lookback=LOOKBACK, max_time_ids=MAX_TIME_IDS_PER_STOCK, num_stocks=NUM_STOCKS)
    # Save model
    torch.save(model.state_dict(), "tinycnn_multi_stock.pth")
    print("Saved model: tinycnn_multi_stock.pth")


Device: cuda
BOOK_PATH: /content/drive/MyDrive/Fuzzy_pool/optiver-realized-volatility-prediction/book_train.parquet
EDA notebook (for reference): /mnt/data/optiver-realized-volatility-prediction-eda.ipynb
Training on stock_count = 112
Epoch 1/10  loss=0.4723 acc=0.7708 time=489.3s
Epoch 2/10  loss=0.4265 acc=0.8020 time=271.1s
Epoch 3/10  loss=0.4244 acc=0.8027 time=264.1s
Epoch 4/10  loss=0.4228 acc=0.8034 time=264.6s
Epoch 5/10  loss=0.4215 acc=0.8038 time=264.5s
Epoch 6/10  loss=0.4203 acc=0.8044 time=262.5s
Epoch 7/10  loss=0.4196 acc=0.8046 time=265.1s
Epoch 8/10  loss=0.4189 acc=0.8048 time=262.8s
Epoch 9/10  loss=0.4183 acc=0.8055 time=266.0s
Epoch 10/10  loss=0.4181 acc=0.8056 time=266.7s
Saved model: tinycnn_multi_stock.pth


In [ ]:
# ================================================================
# Multi-stock Memory-Optimized Streaming Training
# Optiver Market Data + Tiny CNN + Light Fuzzy Pooling
# ================================================================
# Mount Drive first if in Google Colab:
# from google.colab import drive; drive.mount('/content/drive')

import os
from pathlib import Path
import random
import time
import collections
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

# -------------------------
# Config
# -------------------------
DATA_DIR = "/content/drive/MyDrive/Fuzzy_pool/optiver-realized-volatility-prediction"
BOOK_PATH = Path(DATA_DIR) / "book_train.parquet"
TRAIN_CSV = Path(DATA_DIR) / "train.csv"
LOOKBACK = 60
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
MAX_TIME_IDS_PER_STOCK = None
NUM_STOCKS = None
USE_REAL_TARGETS = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)
print("Using TinyCNN + LightFuzzyPooling")

# -------------------------------------------------------
# Helpers
# -------------------------------------------------------
def get_stock_list(train_csv_path, num_stocks=None):
    df = pd.read_csv(train_csv_path)
    ids = sorted(df['stock_id'].unique().tolist())
    if num_stocks: ids = ids[:num_stocks]
    return ids

def read_book_for_stock_pa(book_parquet_path, stock_id, columns=None, max_time_ids=None):
    dataset = ds.dataset(str(book_parquet_path), format="parquet", partitioning="hive")
    table = dataset.to_table(filter=(ds.field("stock_id") == stock_id), columns=columns)
    df = table.to_pandas().sort_values(["time_id", "seconds_in_bucket"]).reset_index(drop=True)
    if df.empty: raise RuntimeError(f"No data for stock_id={stock_id}")
    if max_time_ids:
        tids = df["time_id"].unique()[:max_time_ids]
        df = df[df["time_id"].isin(tids)].reset_index(drop=True)
    return df

# -------------------------------------------------------
# Feature Engineering
# -------------------------------------------------------
def aggregate_book_to_time_features(df):
    df = df.copy()
    denom = (df["bid_size1"] + df["ask_size1"] + 1e-9)
    df["wap1"] = (df["bid_price1"] * df["ask_size1"] + df["ask_price1"] * df["bid_size1"]) / denom
    df["spread1"] = df["ask_price1"] - df["bid_price1"]
    df["mid1"] = (df["ask_price1"] + df["bid_price1"]) / 2.0
    df["liq_imb1"] = (df["bid_size1"] - df["ask_size1"]) / denom

    agg = df.groupby("time_id").agg({
        "wap1": ["mean", "std"],
        "spread1": ["mean", "std"],
        "mid1": ["mean"],
        "liq_imb1": ["mean"]
    })
    agg.columns = ["_".join(col).strip() for col in agg.columns.values]
    return agg.reset_index().sort_values("time_id").reset_index(drop=True)

# -------------------------------------------------------
# Streaming IterableDataset
# -------------------------------------------------------
class StockWindowStream(IterableDataset):
    def __init__(self, book_path, stock_id, lookback, max_time_ids, normalize=True, train_csv=None):
        self.book_path = book_path
        self.stock_id = stock_id
        self.lookback = lookback
        self.max_time_ids = max_time_ids
        self.normalize = normalize
        self.train_csv = train_csv

    def __iter__(self):
        df = read_book_for_stock_pa(self.book_path, self.stock_id, max_time_ids=self.max_time_ids)
        feats = aggregate_book_to_time_features(df)

        # Labels
        if USE_REAL_TARGETS:
            ydf = pd.read_csv(self.train_csv)
            s = ydf[ydf.stock_id == self.stock_id].set_index("time_id")["target"]
            vals = feats["time_id"].map(s).fillna(0.0).values.astype(np.float32)
            med = np.median(vals)
            labels = (vals > med).astype(np.int64)
        else:
            mid = feats["mid1_mean"].values.astype(np.float32)
            next_ret = np.concatenate([mid[1:] - mid[:-1], [0]])
            labels = (next_ret > np.std(next_ret) * 0.01).astype(np.int64)

        cols = [c for c in feats.columns if c != "time_id"]
        arr = feats[cols].values.astype(np.float32)

        dq = collections.deque(maxlen=self.lookback)
        for i, row in enumerate(arr):
            dq.append(row)
            if len(dq) == self.lookback:
                win = np.stack(dq, axis=1)
                if self.normalize:
                    win = (win - win.mean(axis=1, keepdims=True)) / (win.std(axis=1, keepdims=True) + 1e-9)
                yield torch.tensor(win, dtype=torch.float32), int(labels[i - (self.lookback - 1)])

# -------------------------------------------------------
# Light Fuzzy Pooling Layer
# -------------------------------------------------------
class LightFuzzyPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2, num_terms=3):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)
        self.num_terms = num_terms
        self.tfn_logits = nn.Parameter(torch.randn(num_terms, 3) * 0.05)

    def forward(self, x):
        pooled = self.pool(x)
        raw = F.softplus(self.tfn_logits)
        a = raw[:, 0]
        b = a + raw[:, 1]
        c = b + raw[:, 2]

        x_ = pooled.unsqueeze(-1)
        left = torch.where((x_ > a) & (x_ <= b), (x_ - a) / (b - a + 1e-9), 0.0)
        right = torch.where((x_ > b) & (x_ < c), (c - x_) / (c - b + 1e-9), 0.0)
        mu = left + right

        centroids = b.view(1, 1, 1, 1, -1)
        return (mu * centroids).sum(-1) / (mu.sum(-1) + 1e-9)

# -------------------------------------------------------
# TinyCNN + LightFuzzyPooling
# -------------------------------------------------------
class TinyFuzzyCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.fuzzy1 = LightFuzzyPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.fuzzy2 = LightFuzzyPool2d(2, 2)

        self.gpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.fuzzy1(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.fuzzy2(x)

        x = self.gpool(x).view(x.size(0), -1)
        return self.fc(x)

# -------------------------------------------------------
# Collate
# -------------------------------------------------------
def collate_fn(batch):
    windows, labels = zip(*batch)
    xs = torch.stack(windows).unsqueeze(1)
    ys = torch.tensor(labels, dtype=torch.long)
    return xs, ys

# -------------------------------------------------------
# Multi-stock Training Loop
# -------------------------------------------------------
def train_multi_stock():
    stock_ids = get_stock_list(TRAIN_CSV, NUM_STOCKS)
    print("Training on", len(stock_ids), "stocks")

    model = TinyFuzzyCNN(in_channels=1).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()

    for ep in range(1, EPOCHS + 1):
        random.shuffle(stock_ids)
        total_loss = 0
        total_correct = 0
        total = 0
        t0 = time.time()

        for sid in stock_ids:
            ds = StockWindowStream(BOOK_PATH, sid, LOOKBACK, MAX_TIME_IDS_PER_STOCK,
                                   normalize=True, train_csv=TRAIN_CSV)
            loader = DataLoader(ds, batch_size=BATCH_SIZE, collate_fn=collate_fn,
                                num_workers=NUM_WORKERS)

            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)

                preds = model(xb)
                loss = crit(preds, yb)

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item() * xb.size(0)
                total_correct += (preds.argmax(1) == yb).sum().item()
                total += xb.size(0)

        print(f"Epoch {ep}/{EPOCHS} | Loss={total_loss/total:.4f} | Acc={total_correct/total:.4f} | Time={time.time()-t0:.1f}s")

    torch.save(model.state_dict(), "tiny_fuzzy_cnn.pth")
    print("Saved model to tiny_fuzzy_cnn.pth")

    return model

# ----------------------
# Run Training
# ----------------------
if __name__ == "__main__":
    model = train_multi_stock()


Device: cuda
Using TinyCNN + LightFuzzyPooling
Training on 112 stocks
Epoch 1/10 | Loss=0.6409 | Acc=0.6205 | Time=503.9s


In [ ]:
# ================================================================
# Multi-stock Memory-Optimized Streaming Training
# Optiver Market Data + TinyFuzzyCNN + GRU hybrid
# ================================================================
# Mount Drive first if in Google Colab:
# from google.colab import drive; drive.mount('/content/drive')

import os
from pathlib import Path
import random
import time
import collections
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader

# -------------------------
# Config
# -------------------------
DATA_DIR = "/content/drive/MyDrive/Fuzzy_pool/optiver-realized-volatility-prediction"
BOOK_PATH = Path(DATA_DIR) / "book_train.parquet"
TRAIN_CSV = Path(DATA_DIR) / "train.csv"
LOOKBACK = 60
BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
MAX_TIME_IDS_PER_STOCK = None
NUM_STOCKS = None
USE_REAL_TARGETS = True
EDA_NOTEBOOK_PATH = "/mnt/data/optiver-realized-volatility-prediction-eda.ipynb"  # your uploaded notebook path (local)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)
print("Using TinyFuzzyCNN + GRU hybrid")
print("EDA notebook (for reference):", EDA_NOTEBOOK_PATH)

# -------------------------------------------------------
# Helpers
# -------------------------------------------------------
def get_stock_list(train_csv_path, num_stocks=None):
    df = pd.read_csv(train_csv_path)
    ids = sorted(df['stock_id'].unique().tolist())
    if num_stocks:
        ids = ids[:num_stocks]
    return ids

def read_book_for_stock_pa(book_parquet_path, stock_id, columns=None, max_time_ids=None):
    dataset = ds.dataset(str(book_parquet_path), format="parquet", partitioning="hive")
    table = dataset.to_table(filter=(ds.field("stock_id") == stock_id), columns=columns)
    df = table.to_pandas().sort_values(["time_id", "seconds_in_bucket"]).reset_index(drop=True)
    if df.empty:
        raise RuntimeError(f"No data for stock_id={stock_id}")
    if max_time_ids:
        tids = df["time_id"].unique()[:max_time_ids]
        df = df[df["time_id"].isin(tids)].reset_index(drop=True)
    return df

def aggregate_book_to_time_features(df):
    df = df.copy()
    denom = (df["bid_size1"] + df["ask_size1"] + 1e-9)
    df["wap1"] = (df["bid_price1"] * df["ask_size1"] + df["ask_price1"] * df["bid_size1"]) / denom
    df["spread1"] = df["ask_price1"] - df["bid_price1"]
    df["mid1"] = (df["ask_price1"] + df["bid_price1"]) / 2.0
    df["liq_imb1"] = (df["bid_size1"] - df["ask_size1"]) / denom

    agg = df.groupby("time_id").agg({
        "wap1": ["mean", "std"],
        "spread1": ["mean", "std"],
        "mid1": ["mean"],
        "liq_imb1": ["mean"]
    })
    agg.columns = ["_".join(col).strip() for col in agg.columns.values]
    return agg.reset_index().sort_values("time_id").reset_index(drop=True)

def determine_num_features_sample(book_path, stock_id, max_time_ids=50):
    """
    Read a small sample from parquet and return number of feature rows (height H).
    """
    df = read_book_for_stock_pa(book_path, stock_id, max_time_ids=max_time_ids)
    feats = aggregate_book_to_time_features(df)
    cols = [c for c in feats.columns if c != "time_id"]
    return len(cols)

# -------------------------------------------------------
# Streaming IterableDataset
# -------------------------------------------------------
class StockWindowStream(IterableDataset):
    def __init__(self, book_path, stock_id, lookback, max_time_ids, normalize=True, train_csv=None):
        self.book_path = book_path
        self.stock_id = stock_id
        self.lookback = lookback
        self.max_time_ids = max_time_ids
        self.normalize = normalize
        self.train_csv = train_csv

    def __iter__(self):
        df = read_book_for_stock_pa(self.book_path, self.stock_id, max_time_ids=self.max_time_ids)
        feats = aggregate_book_to_time_features(df)

        # Labels
        if self.train_csv is not None and USE_REAL_TARGETS:
            ydf = pd.read_csv(self.train_csv)
            s = ydf[ydf.stock_id == self.stock_id].set_index("time_id")["target"]
            vals = feats["time_id"].map(s).fillna(0.0).values.astype(np.float32)
            med = np.median(vals)
            labels = (vals > med).astype(np.int64)
        else:
            mid = feats["mid1_mean"].values.astype(np.float32)
            next_ret = np.concatenate([mid[1:] - mid[:-1], [0]])
            labels = (next_ret > np.std(next_ret) * 0.01).astype(np.int64)

        cols = [c for c in feats.columns if c != "time_id"]
        arr = feats[cols].values.astype(np.float32)

        dq = collections.deque(maxlen=self.lookback)
        for i, row in enumerate(arr):
            dq.append(row)
            if len(dq) == self.lookback:
                win = np.stack(dq, axis=1)  # (n_features, lookback)
                if self.normalize:
                    win = (win - win.mean(axis=1, keepdims=True)) / (win.std(axis=1, keepdims=True) + 1e-9)
                yield torch.tensor(win, dtype=torch.float32), int(labels[i - (self.lookback - 1)])

# -------------------------------------------------------
# Light Fuzzy Pooling Layer (pool along width/time by default)
# Accepts tuple kernel_size and stride to allow (height_pool, width_pool)
# -------------------------------------------------------
class LightFuzzyPool2d(nn.Module):
    def __init__(self, kernel_size=(1,2), stride=(1,2), num_terms=3):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)
        self.num_terms = num_terms
        self.tfn_logits = nn.Parameter(torch.randn(num_terms, 3) * 0.05)

    def forward(self, x):
        # x: (B,C,H,W)
        pooled = self.pool(x)  # reduce width (time) by default
        raw = F.softplus(self.tfn_logits)
        a = raw[:, 0]
        b = a + raw[:, 1]
        c = b + raw[:, 2]

        # broadcast shapes
        x_ = pooled.unsqueeze(-1)  # B,C,H,W,1
        a_ = a.view(1,1,1,1,-1)
        b_ = b.view(1,1,1,1,-1)
        c_ = c.view(1,1,1,1,-1)

        left = torch.where((x_ > a_) & (x_ <= b_), (x_ - a_) / (b_ - a_ + 1e-9), torch.zeros_like(x_))
        right = torch.where((x_ > b_) & (x_ < c_), (c_ - x_) / (c_ - b_ + 1e-9), torch.zeros_like(x_))
        mu = left + right  # B,C,H,W,terms

        # centroid is b
        centroids = b.view(1,1,1,1,-1)
        out = (mu * centroids).sum(-1) / (mu.sum(-1) + 1e-9)  # B,C,H,W
        return out

# -------------------------------------------------------
# TinyFuzzyCNN + GRU hybrid
# - conv layers extract features (H preserved)
# - fuzzy pools reduce TIME dimension (W reduces -> seq_len)
# - prepare (B, seq_len, feat_dim) and feed to GRU
# -------------------------------------------------------
class TinyFuzzyGRU(nn.Module):
    def __init__(self, in_channels=1, n_features=6, conv_channels=[16, 32], gru_hidden=64, gru_layers=1, bidir=True, num_classes=2):
        super().__init__()
        self.n_features = n_features  # height H (number of indicators)
        self.conv1 = nn.Conv2d(in_channels, conv_channels[0], kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(conv_channels[0])
        # pool reduces time (W) only: kernel=(1,2), stride=(1,2)
        self.fuzzy1 = LightFuzzyPool2d(kernel_size=(1,2), stride=(1,2))

        self.conv2 = nn.Conv2d(conv_channels[0], conv_channels[1], kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(conv_channels[1])
        self.fuzzy2 = LightFuzzyPool2d(kernel_size=(1,2), stride=(1,2))

        # after convs/pooling, height remains approx H' = ceil(H/1) because pooling only along width
        # GRU input size = conv_channels[1] * H (we preserve H by not pooling in height)
        self.gru_input_size = conv_channels[1] * self.n_features
        self.gru_hidden = gru_hidden
        self.bidir = bidir
        self.gru = nn.GRU(input_size=self.gru_input_size, hidden_size=gru_hidden, num_layers=gru_layers,
                          batch_first=True, bidirectional=bidir)

        self.out_feat = gru_hidden * (2 if bidir else 1)
        self.fc = nn.Linear(self.out_feat, num_classes)

    def forward(self, x):
        # x: (B,1,H,W)
        x = F.relu(self.bn1(self.conv1(x)))    # (B, C1, H, W)
        x = self.fuzzy1(x)                     # (B, C1, H, W')

        x = F.relu(self.bn2(self.conv2(x)))    # (B, C2, H, W')
        x = self.fuzzy2(x)                     # (B, C2, H, W'')

        B, C, H, W = x.size()
        # prepare sequence over width dimension (time)
        # permute to (B, W, C, H) -> reshape to (B, W, C*H)
        seq = x.permute(0, 3, 1, 2).contiguous().view(B, W, C * H)

        # GRU expects (B, seq_len, input_size)
        out, h = self.gru(seq)  # out: (B, seq_len, num_directions*hidden), h: (num_layers*num_directions, B, hidden)
        # take last timestep's output (out[:, -1, :]) or derive from h
        last = out[:, -1, :]  # (B, out_feat)
        logits = self.fc(last)
        return logits

# -------------------------------------------------------
# Collate
# -------------------------------------------------------
def collate_fn(batch):
    windows, labels = zip(*batch)
    xs = torch.stack(windows).unsqueeze(1)  # B,1,H,W
    ys = torch.tensor(labels, dtype=torch.long)
    return xs, ys

# -------------------------------------------------------
# Multi-stock Training Loop
# -------------------------------------------------------
def train_multi_stock():
    stock_ids = get_stock_list(TRAIN_CSV, NUM_STOCKS)
    print("Training on", len(stock_ids), "stocks")

    # determine n_features automatically from first stock sample (safe)
    sample_stock = stock_ids[0]
    n_features = determine_num_features_sample(BOOK_PATH, sample_stock, max_time_ids=50)
    print("Detected n_features (height H) =", n_features)

    model = TinyFuzzyGRU(in_channels=1, n_features=n_features, conv_channels=[16,32],
                         gru_hidden=64, gru_layers=1, bidir=True, num_classes=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()

    for ep in range(1, EPOCHS + 1):
        random.shuffle(stock_ids)
        total_loss = 0.0
        total_correct = 0
        total = 0
        t0 = time.time()

        for sid in stock_ids:
            ds = StockWindowStream(BOOK_PATH, sid, LOOKBACK, MAX_TIME_IDS_PER_STOCK,
                                   normalize=True, train_csv=TRAIN_CSV if USE_REAL_TARGETS else None)
            loader = DataLoader(ds, batch_size=BATCH_SIZE, collate_fn=collate_fn, num_workers=NUM_WORKERS)

            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb)
                loss = crit(preds, yb)

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item() * xb.size(0)
                total_correct += (preds.argmax(1) == yb).sum().item()
                total += xb.size(0)

        if total == 0:
            print("No samples found. Check LOOKBACK / MAX_TIME_IDS_PER_STOCK.")
            break

        print(f"Epoch {ep}/{EPOCHS} | Loss={total_loss/total:.4f} | Acc={total_correct/total:.4f} | Time={time.time()-t0:.1f}s")

    torch.save(model.state_dict(), "tiny_fuzzy_gru.pth")
    print("Saved model to tiny_fuzzy_gru.pth")
    return model

# ----------------------
# Run Training
# ----------------------
if __name__ == "__main__":
    model = train_multi_stock()


Device: cuda
Using TinyFuzzyCNN + GRU hybrid
EDA notebook (for reference): /mnt/data/optiver-realized-volatility-prediction-eda.ipynb
Training on 112 stocks
Detected n_features (height H) = 6
Epoch 1/10 | Loss=0.4829 | Acc=0.7561 | Time=473.1s
Epoch 2/10 | Loss=0.4316 | Acc=0.7968 | Time=294.0s
Epoch 3/10 | Loss=0.4202 | Acc=0.8036 | Time=291.2s
Epoch 4/10 | Loss=0.4152 | Acc=0.8057 | Time=291.1s
Epoch 5/10 | Loss=0.4104 | Acc=0.8086 | Time=291.4s
Epoch 6/10 | Loss=0.4064 | Acc=0.8100 | Time=290.4s
Epoch 7/10 | Loss=0.4021 | Acc=0.8125 | Time=290.7s
Epoch 8/10 | Loss=0.3972 | Acc=0.8151 | Time=289.5s
Epoch 9/10 | Loss=0.3925 | Acc=0.8182 | Time=290.7s
Epoch 10/10 | Loss=0.3888 | Acc=0.8196 | Time=294.6s
Saved model to tiny_fuzzy_gru.pth
